<a href="https://colab.research.google.com/github/abdullahhadi9898/-indata-software-sentiment/blob/main/indata_sentiment_analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
#importing data from google drive to collab

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# Cell 4 — Verify files are accessible
import os

# Update this path to match your actual folder name in Google Drive
DRIVE_PATH = '/content/drive/MyDrive/indata-project'

# List files in the folder
files = os.listdir(DRIVE_PATH)
print("Files found in your Google Drive folder:")
for f in files:
    size = os.path.getsize(f'{DRIVE_PATH}/{f}') / (1024**3)
    print(f"  {f}  —  {size:.2f} GB")

Files found in your Google Drive folder:
  meta_Software.jsonl.gz  —  0.06 GB
  Software.jsonl.gz  —  0.46 GB


In [27]:
# Cell 5 — Set file paths
REVIEW_FILE = '/content/drive/MyDrive/indata-project/Software.jsonl.gz'
META_FILE = '/content/drive/MyDrive/indata-project/meta_Software.jsonl.gz'
SAMPLE_SIZE = 20000  # 20k per star rating = 100k total

print("✅ File paths set")
print(f"   Review file : {REVIEW_FILE}")
print(f"   Meta file   : {META_FILE}")

✅ File paths set
   Review file : /content/drive/MyDrive/indata-project/Software.jsonl.gz
   Meta file   : /content/drive/MyDrive/indata-project/meta_Software.jsonl.gz


In [28]:
# Cell 6 — Load raw review data
import json
import gzip

print("Loading reviews... this will take a few minutes.")
print("You will see progress updates every 500,000 rows.\n")

reviews = []
with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            reviews.append(json.loads(line.strip()))
        except json.JSONDecodeError:
            continue

        if i % 500000 == 0 and i > 0:
            print(f"  Scanned {i:,} rows so far...")

df_raw = pd.DataFrame(reviews)

print(f"\n✅ Raw data loaded successfully")
print(f"   Total rows    : {df_raw.shape[0]:,}")
print(f"   Total columns : {df_raw.shape[1]}")
print(f"\nColumns found:")
print(df_raw.columns.tolist())

Loading reviews... this will take a few minutes.
You will see progress updates every 500,000 rows.

  Scanned 500,000 rows so far...
  Scanned 1,000,000 rows so far...
  Scanned 1,500,000 rows so far...
  Scanned 2,000,000 rows so far...
  Scanned 2,500,000 rows so far...
  Scanned 3,000,000 rows so far...
  Scanned 3,500,000 rows so far...
  Scanned 4,000,000 rows so far...
  Scanned 4,500,000 rows so far...


NameError: name 'pd' is not defined

importing libraries

In [30]:
# Cell 6 (final fix) — Install + Import + Load all in one cell
import subprocess
subprocess.run(['pip', 'install', 'vaderSentiment', 'gensim', 'wordcloud', '-q'])

import json
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

# File paths
REVIEW_FILE = '/content/drive/MyDrive/indata-project/Software.jsonl.gz'
META_FILE   = '/content/drive/MyDrive/indata-project/meta_Software.jsonl.gz'
SAMPLE_SIZE = 20000

print("✅ Libraries installed and imported")
print("Loading reviews now... do not click anything\n")

reviews = []
with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            reviews.append(json.loads(line.strip()))
        except json.JSONDecodeError:
            continue
        if i % 500000 == 0 and i > 0:
            print(f"  Scanned {i:,} rows so far...")

df_raw = pd.DataFrame(reviews)

print(f"\n✅ Data loaded")
print(f"   Total rows    : {df_raw.shape[0]:,}")
print(f"   Total columns : {df_raw.shape[1]}")
print(f"\nColumns:")
print(df_raw.columns.tolist())

✅ Libraries installed and imported
Loading reviews now... do not click anything

  Scanned 500,000 rows so far...
  Scanned 1,000,000 rows so far...
  Scanned 1,500,000 rows so far...
  Scanned 2,000,000 rows so far...
  Scanned 2,500,000 rows so far...
  Scanned 3,000,000 rows so far...
  Scanned 3,500,000 rows so far...
  Scanned 4,000,000 rows so far...
  Scanned 4,500,000 rows so far...

✅ Data loaded
   Total rows    : 4,880,181
   Total columns : 10

Columns:
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
